# 05 · Landmark preprocessing and feature sequences

Normalizes each clip by shoulder centre and width, fills gaps of up to 3 frames, and resamples it to 32 frames. It saves raw coordinates, velocity, and joint angles. Remaining missing values stay marked as NaN and are filled within each model training fold.


In [ ]:
from pathlib import Path
import runpy
ROOT = Path.cwd().resolve()
while not (ROOT / "data/processed/WLASL100/all_metadata.csv").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
script = ROOT / "scripts" / "preprocess_landmarks.py"
if not script.exists():
    raise FileNotFoundError(script)
runpy.run_path(str(script), run_name="__main__")


## Files created

The script saves `sequences_v1.npz`, `manifest.csv`, and `summary.json` under `data/processed/WLASL100/preprocessed/`. Rows in the arrays and manifest share the same order, with each video matched to its gloss, signer, and saved split.


## Quick check

The table below summarizes how many clips were processed and how much landmark data is still missing after preprocessing.

In [ ]:
import json
import pandas as pd

preprocessed_dir = ROOT / 'data/processed/WLASL100/preprocessed'
summary = json.loads((preprocessed_dir / 'summary.json').read_text())
print(f"Processed {summary['n_videos']} clips into {summary['frames']} frames each.")
manifest = pd.read_csv(preprocessed_dir / 'manifest.csv')
display(manifest.groupby('split').agg(
    clips=('video_id', 'count'),
    median_missing_before_pct=('missing_before_pct', 'median'),
    median_missing_after_pct=('missing_after_pct', 'median'),
).round(1))